# NB3 · Non-linearità, ensemble e il caso retail Conad

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/battles5/conad-statistical-learning/blob/main/notebooks/NB3_ensemble_caso_retail.ipynb)

**albero, random forest, boosting e SVM su un dataset retail sintetico**

corso *Apprendimento statistico* per CONAD Nord Ovest · Orso Peruzzi (IFAB)

> come si esegue una cella: clicca dentro e premi **Shift + Invio**. esegui le celle in ordine, dall'alto verso il basso.

## 1. Il dataset retail (sintetico, stile Conad)

10.000 clienti di una carta fedelta'. due compiti sugli **stessi dati**:

- **churn** (classificazione): il cliente e' a rischio abbandono? (0/1);
- **spesa prevista a 12 mesi** (regressione): quanto spenderà? (euro).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, r2_score

df = pd.read_csv("https://raw.githubusercontent.com/battles5/conad-statistical-learning/main/data/conad_retail.csv")
print("righe:", len(df), " colonne:", df.shape[1])
df.head()

In [ ]:
# prepariamo le variabili: togliamo l'id e codifichiamo le categoriche
cat = ["formato_pdv", "canale", "reparto_preferito"]
X = pd.get_dummies(df.drop(columns=["cliente_id", "churn", "spesa_prevista_12m"]), columns=cat)
y_churn = df["churn"]
y_spesa = df["spesa_prevista_12m"]
print("numero di feature dopo la codifica:", X.shape[1])

## 2. Compito A, churn: un albero che overfitta

l'albero di decisione e' intuitivo, ma se lo lasciamo crescere troppo impara il rumore: ottimo sul training, peggiore sul test.

> **manopola**: cambia `PROFONDITA` (prova 2, 4, 8, 20) e confronta AUC di training e di test.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

Xtr, Xte, ytr, yte = train_test_split(X, y_churn, test_size=0.30, random_state=0, stratify=y_churn)

# >>> MANOPOLA: profondità massima dell'albero <<<
PROFONDITA = 4

albero = DecisionTreeClassifier(max_depth=PROFONDITA, random_state=0).fit(Xtr, ytr)
auc_tr = roc_auc_score(ytr, albero.predict_proba(Xtr)[:, 1])
auc_te = roc_auc_score(yte, albero.predict_proba(Xte)[:, 1])
print(f"profondità {PROFONDITA}  ->  AUC training: {auc_tr:.3f}   AUC test: {auc_te:.3f}")
print("differenza training - test (più è grande, più overfitta):", round(auc_tr - auc_te, 3))

prova `PROFONDITA = 20`: l'AUC di training si avvicina a 1 ma quella di test cala. e' overfitting.

## 3. La random forest stabilizza

tanti alberi diversi mediati insieme: meno varianza, di solito molto piu' accurata. e ci dice quali variabili contano.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1).fit(Xtr, ytr)
print("AUC test random forest:", round(roc_auc_score(yte, rf.predict_proba(Xte)[:, 1]), 3))

imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)
plt.figure(figsize=(7.5, 4.5))
imp[::-1].plot(kind="barh", color="#00ADCF")
plt.title("Random forest: importanza delle variabili (churn)"); plt.tight_layout(); plt.show()

le variabili in cima (recency, visite, distanza, reclami) sono i veri segnali del churn; in fondo trovi le feature di rumore.

## 4. Gradient boosting

alberi piccoli in sequenza, ognuno corregge gli errori del precedente: spesso il piu' accurato.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=0).fit(Xtr, ytr)
print("AUC test gradient boosting:", round(roc_auc_score(yte, gb.predict_proba(Xte)[:, 1]), 3))

## 5. SVM: il confine in 2D

per vedere il confine di decisione usiamo solo due variabili (recency e visite) e una SVM con kernel non lineare.

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

due = ["recency_giorni", "n_visite_mese"]
X2 = df[due].values
X2tr, X2te, y2tr, y2te = train_test_split(X2, y_churn, test_size=0.30, random_state=0, stratify=y_churn)
svm = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0)).fit(X2tr, y2tr)

xx, yy = np.meshgrid(np.linspace(0, 120, 300), np.linspace(0, 20, 300))
Z = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
plt.figure(figsize=(7.5, 5))
plt.contourf(xx, yy, Z, alpha=0.25, cmap="coolwarm")
plt.scatter(X2te[:, 0], X2te[:, 1], c=y2te, cmap="coolwarm", s=10, alpha=0.5)
plt.xlabel("recency (giorni dall'ultimo acquisto)"); plt.ylabel("visite al mese")
plt.title("SVM: confine di decisione per il churn"); plt.show()

## 6. Compito B, previsione vendite: lineare vs boosting

stesso dataset, target continuo. qui si vede che i modelli flessibili battono il lineare grazie alle non-linearita'.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor

Xtr, Xte, str_, ste = train_test_split(X, y_spesa, test_size=0.30, random_state=0)
lin = LinearRegression().fit(Xtr, str_)
gbr = GradientBoostingRegressor(random_state=0).fit(Xtr, str_)
print("R2 test lineare:          ", round(r2_score(ste, lin.predict(Xte)), 3))
print("R2 test gradient boosting:", round(r2_score(ste, gbr.predict(Xte)), 3))

**quale modello per quale obiettivo?** se conta spiegare, parti dal lineare interpretabile; se conta solo prevedere bene, gli ensemble di solito vincono. e' di nuovo il trade-off flessibilita' / interpretabilita'.

---

### Cella bonus: permutation importance

l'importanza basata sull'impurita' favorisce le variabili continue. la *permutation importance* e' piu' onesta: mescola una colonna alla volta e misura quanto peggiora il modello. le feature di rumore vanno vicino a zero.

In [ ]:
# BONUS
from sklearn.inspection import permutation_importance

Xtr, Xte, ytr, yte = train_test_split(X, y_churn, test_size=0.30, random_state=0, stratify=y_churn)
rf = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1).fit(Xtr, ytr)
pi = permutation_importance(rf, Xte, yte, n_repeats=5, random_state=0, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=X.columns).sort_values(ascending=False).head(10)
imp[::-1].plot(kind="barh", figsize=(7.5, 4.5), color="#002060")
plt.title("Permutation importance (churn)"); plt.tight_layout(); plt.show()